### Eurostat Agricultural Employment Data Collection

This notebook retrieves county-level employment data from Eurostat and prepares agricultural employment indicators for the 2012–2022 analysis period.

**Source:** Eurostat  
**Dataset:** `nama_10r_3empers`  
**Geographical level:** NUTS 3 (Romanian counties)  
**Period:** 2012–2022

The workflow includes:
- retrieving employment data from Eurostat;
- filtering Romanian NUTS 3 regions;
- extracting total employment and employment in agriculture, forestry and fishing (AFF);
- calculating the share of employment in AFF;
- harmonizing NUTS 3 codes with Romanian county names;
- constructing annual changes in agricultural employment;
- validating the resulting county-year panel.

In [5]:
import numpy as np
import pandas as pd

### 1. Data Retrieval and Initial Preparation

In [6]:
import eurostat
import pandas as pd

code = "nama_10r_3empers"

# descarcă tot datasetul în format pandas
df_euro = eurostat.get_data_df(code)

df_euro.head()

,freq,unit,wstatus,nace_r2,geo\TIME_PERIOD,2000,2001,2002,2003,2004,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,A,THS,EMP,A,AT,230.7,229.6,225.5,224.9,218.0,...,176.0,171.3,167.0,156.3,148.7,151.5,155.4,150.0,138.1,136.5
1,A,THS,EMP,A,AT1,71.0,70.1,68.6,68.0,65.6,...,53.2,52.3,51.4,48.2,46.3,46.6,47.8,46.5,42.9,42.3
2,A,THS,EMP,A,AT11,13.1,12.8,12.4,12.2,11.6,...,8.5,8.3,8.3,7.8,7.7,7.8,8.1,7.9,7.3,7.1
3,A,THS,EMP,A,AT111,1.8,1.7,1.6,1.6,1.5,...,0.9,0.9,0.9,0.9,0.8,0.8,0.8,0.8,0.8,NaN
4,A,THS,EMP,A,AT112,6.3,6.1,6.0,5.9,5.6,...,4.5,4.4,4.4,4.2,4.1,4.2,4.3,4.1,3.9,NaN


In [7]:
df_euro = df_euro.rename(columns={r"geo\TIME_PERIOD": "geo"})

In [8]:
df_ro = df_euro[
    (df_euro["geo"].str.startswith("RO", na=False)) &
    (df_euro["wstatus"] == "EMP") &
    (df_euro["nace_r2"].isin(["A", "TOTAL"]))
].copy()

In [9]:
year_cols = [col for col in df_ro.columns if str(col).isdigit()]
year_cols = [col for col in year_cols if 2012 <= int(col) <= 2022]

df_ro = df_ro[["unit", "wstatus", "nace_r2", "geo"] + year_cols]

In [10]:
df_ro_long = df_ro.melt(
    id_vars=["unit", "wstatus", "nace_r2", "geo"],
    value_vars=year_cols,
    var_name="an",
    value_name="employment_thousand"
)

df_ro_long["an"] = df_ro_long["an"].astype(int)

In [11]:
df_ro_wide = df_ro_long.pivot_table(
    index=["geo", "an"],
    columns="nace_r2",
    values="employment_thousand",
    aggfunc="first"
).reset_index()

df_ro_wide = df_ro_wide.rename(columns={
    "A": "employment_aff_thousand",
    "TOTAL": "employment_total_thousand"
})

df_ro_wide["share_aff"] = (
    df_ro_wide["employment_aff_thousand"] /
    df_ro_wide["employment_total_thousand"]
) * 100

df_ro_wide.head()

nace_r2,geo,an,employment_aff_thousand,employment_total_thousand,share_aff
0,RO,2012,2649.4,8645.3,30.645553
1,RO,2013,2591.4,8569.4,30.240157
2,RO,2014,2528.4,8634.6,29.282190
3,RO,2015,2251.1,8525.7,26.403697
4,RO,2016,2009.5,8429.6,23.838616


### 2. NUTS 3 Filtering and Data Validation

In [12]:
euro_check = df_ro_wide.groupby("an")[[
    "employment_aff_thousand",
    "employment_total_thousand"
]].sum()

euro_check["share_aff"] = (
    euro_check["employment_aff_thousand"] /
    euro_check["employment_total_thousand"]
) * 100

euro_check

nace_r2,employment_aff_thousand,employment_total_thousand,share_aff
an,,,
2012,10597.64,34581.21,30.645660
2013,10365.68,34277.65,30.240346
2014,10113.63,34538.45,29.282235
2015,9004.44,34102.84,26.403783
2016,8038.05,33718.46,23.838722
2017,8063.23,34524.82,23.354879
2018,8051.22,34555.20,23.299590
2019,7712.43,34597.98,22.291562
2020,7309.23,33888.41,21.568524


In [13]:
df_ro_wide["geo"].nunique()

55

In [14]:
sorted(df_ro_wide["geo"].unique())

['RO',
 'RO1',
 'RO11',
 'RO111',
 'RO112',
 'RO113',
 'RO114',
 'RO115',
 'RO116',
 'RO12',
 'RO121',
 'RO122',
 'RO123',
 'RO124',
 'RO125',
 'RO126',
 'RO2',
 'RO21',
 'RO211',
 'RO212',
 'RO213',
 'RO214',
 'RO215',
 'RO216',
 'RO22',
 'RO221',
 'RO222',
 'RO223',
 'RO224',
 'RO225',
 'RO226',
 'RO3',
 'RO31',
 'RO311',
 'RO312',
 'RO313',
 'RO314',
 'RO315',
 'RO316',
 'RO317',
 'RO32',
 'RO321',
 'RO322',
 'RO4',
 'RO41',
 'RO411',
 'RO412',
 'RO413',
 'RO414',
 'RO415',
 'RO42',
 'RO421',
 'RO422',
 'RO423',
 'RO424']

In [15]:
df_ro_wide.groupby("geo")["an"].nunique().sort_values()

geo
RO       11
RO1      11
RO11     11
RO111    11
RO112    11
RO113    11
RO114    11
RO115    11
RO116    11
RO12     11
RO121    11
RO122    11
RO123    11
RO124    11
RO125    11
RO126    11
RO2      11
RO21     11
RO211    11
RO212    11
RO213    11
RO214    11
RO215    11
RO216    11
RO22     11
RO221    11
RO222    11
RO223    11
RO224    11
RO225    11
RO226    11
RO3      11
RO31     11
RO311    11
RO312    11
RO313    11
RO314    11
RO315    11
RO316    11
RO317    11
RO32     11
RO321    11
RO322    11
RO4      11
RO41     11
RO411    11
RO412    11
RO413    11
RO414    11
RO415    11
RO42     11
RO421    11
RO422    11
RO423    11
RO424    11
Name: an, dtype: int64

In [16]:
df_euro_nuts3 = df_ro_wide[
    df_ro_wide["geo"].str.match(r"^RO\d{3}$", na=False)
].copy()

df_euro_nuts3["geo"].nunique()

42

In [17]:
sorted(df_euro_nuts3["geo"].unique())

['RO111',
 'RO112',
 'RO113',
 'RO114',
 'RO115',
 'RO116',
 'RO121',
 'RO122',
 'RO123',
 'RO124',
 'RO125',
 'RO126',
 'RO211',
 'RO212',
 'RO213',
 'RO214',
 'RO215',
 'RO216',
 'RO221',
 'RO222',
 'RO223',
 'RO224',
 'RO225',
 'RO226',
 'RO311',
 'RO312',
 'RO313',
 'RO314',
 'RO315',
 'RO316',
 'RO317',
 'RO321',
 'RO322',
 'RO411',
 'RO412',
 'RO413',
 'RO414',
 'RO415',
 'RO421',
 'RO422',
 'RO423',
 'RO424']

In [18]:
df_euro_nuts3 = df_euro_nuts3[df_euro_nuts3["geo"] != "RO321"].copy()

In [19]:
df_euro_nuts3["geo"].nunique()

41

In [20]:
df_euro_nuts3.groupby("geo")["an"].nunique().sort_values()

geo
RO111    11
RO112    11
RO113    11
RO114    11
RO115    11
RO116    11
RO121    11
RO122    11
RO123    11
RO124    11
RO125    11
RO126    11
RO211    11
RO212    11
RO213    11
RO214    11
RO215    11
RO216    11
RO221    11
RO222    11
RO223    11
RO224    11
RO225    11
RO226    11
RO311    11
RO312    11
RO313    11
RO314    11
RO315    11
RO316    11
RO317    11
RO322    11
RO411    11
RO412    11
RO413    11
RO414    11
RO415    11
RO421    11
RO422    11
RO423    11
RO424    11
Name: an, dtype: int64

In [21]:
euro_check_nuts3 = df_euro_nuts3.groupby("an")[[
    "employment_aff_thousand",
    "employment_total_thousand"
]].sum()

euro_check_nuts3["share_aff"] = (
    euro_check_nuts3["employment_aff_thousand"] /
    euro_check_nuts3["employment_total_thousand"]
) * 100

euro_check_nuts3

nace_r2,employment_aff_thousand,employment_total_thousand,share_aff
an,,,
2012,2640.53,7647.47,34.528151
2013,2588.24,7590.70,34.097514
2014,2521.97,7633.41,33.038576
2015,2245.58,7504.92,29.921438
2016,2003.09,7396.89,27.080165
2017,2011.21,7595.68,26.478340
2018,2008.18,7584.83,26.476269
2019,1923.61,7569.83,25.411535
2020,1820.15,7396.17,24.609359


In [22]:
df_euro_nuts3[[
    "employment_aff_thousand",
    "employment_total_thousand",
    "share_aff"
]].isna().sum()

nace_r2
employment_aff_thousand      0
employment_total_thousand    0
share_aff                    0
dtype: int64

In [23]:
sorted(df_euro_nuts3["geo"].unique())

['RO111',
 'RO112',
 'RO113',
 'RO114',
 'RO115',
 'RO116',
 'RO121',
 'RO122',
 'RO123',
 'RO124',
 'RO125',
 'RO126',
 'RO211',
 'RO212',
 'RO213',
 'RO214',
 'RO215',
 'RO216',
 'RO221',
 'RO222',
 'RO223',
 'RO224',
 'RO225',
 'RO226',
 'RO311',
 'RO312',
 'RO313',
 'RO314',
 'RO315',
 'RO316',
 'RO317',
 'RO322',
 'RO411',
 'RO412',
 'RO413',
 'RO414',
 'RO415',
 'RO421',
 'RO422',
 'RO423',
 'RO424']

### 3. County Name Harmonization and Variable Preparation

In [24]:
nuts3_to_judet = {
    # Nord-Vest
    "RO111": "Bihor",
    "RO112": "Bistrita-Nasaud",
    "RO113": "Cluj",
    "RO114": "Maramures",
    "RO115": "Satu Mare",
    "RO116": "Salaj",

    # Centru
    "RO121": "Alba",
    "RO122": "Brasov",
    "RO123": "Covasna",
    "RO124": "Harghita",
    "RO125": "Mures",
    "RO126": "Sibiu",

    # Nord-Est
    "RO211": "Bacau",
    "RO212": "Botosani",
    "RO213": "Iasi",
    "RO214": "Neamt",
    "RO215": "Suceava",
    "RO216": "Vaslui",

    # Sud-Est
    "RO221": "Braila",
    "RO222": "Buzau",
    "RO223": "Constanta",
    "RO224": "Galati",
    "RO225": "Tulcea",
    "RO226": "Vrancea",

    # Sud-Muntenia
    "RO311": "Arges",
    "RO312": "Calarasi",
    "RO313": "Dambovita",
    "RO314": "Giurgiu",
    "RO315": "Ialomita",
    "RO316": "Prahova",
    "RO317": "Teleorman",

    # Bucuresti-Ilfov
    "RO322": "Ilfov",

    # Sud-Vest Oltenia
    "RO411": "Dolj",
    "RO412": "Gorj",
    "RO413": "Mehedinti",
    "RO414": "Olt",
    "RO415": "Valcea",

    # Vest
    "RO421": "Arad",
    "RO422": "Caras-Severin",
    "RO423": "Hunedoara",
    "RO424": "Timis"
}

In [25]:
df_euro_nuts3["judet"] = df_euro_nuts3["geo"].map(nuts3_to_judet)

In [26]:
df_euro_nuts3[df_euro_nuts3["judet"].isna()][["geo"]].drop_duplicates()

nace_r2,geo


In [27]:
df_euro_nuts3.columns.tolist()

['geo',
 'an',
 'employment_aff_thousand',
 'employment_total_thousand',
 'share_aff',
 'judet']

In [28]:
df_euro_nuts3 = df_euro_nuts3.rename(columns={
    "employment_aff_thousand": "ocupati_aff_eurostat_mii",
    "employment_total_thousand": "ocupati_total_eurostat_mii",
    "share_aff": "pondere_ocupati_aff_eurostat"
})

In [29]:
df_euro_nuts3["ocupati_aff_eurostat_nr"] = (
    df_euro_nuts3["ocupati_aff_eurostat_mii"] * 1000
)

df_euro_nuts3["ocupati_total_eurostat_nr"] = (
    df_euro_nuts3["ocupati_total_eurostat_mii"] * 1000
)

In [30]:
df_euro_nuts3.head(3)

nace_r2,geo,an,ocupati_aff_eurostat_mii,ocupati_total_eurostat_mii,pondere_ocupati_aff_eurostat,judet,ocupati_aff_eurostat_nr,ocupati_total_eurostat_nr
33,RO111,2012,76.37,283.64,26.924975,Bihor,76370.0,283640.0
34,RO111,2013,80.05,288.91,27.707591,Bihor,80050.0,288910.0
35,RO111,2014,77.41,296.60,26.099123,Bihor,77410.0,296600.0


In [31]:
df_euro_nuts3.shape

(451, 8)

In [32]:
df_euro_nuts3[[
    "judet",
    "an",
    "ocupati_aff_eurostat_mii",
    "ocupati_total_eurostat_mii",
    "pondere_ocupati_aff_eurostat",
    "ocupati_aff_eurostat_nr",
    "ocupati_total_eurostat_nr"
]].isna().sum()

nace_r2
judet                           0
an                              0
ocupati_aff_eurostat_mii        0
ocupati_total_eurostat_mii      0
pondere_ocupati_aff_eurostat    0
ocupati_aff_eurostat_nr         0
ocupati_total_eurostat_nr       0
dtype: int64

In [33]:
df_euro_nuts3["judet"].nunique()

41

In [34]:
df_euro_nuts3.groupby("judet")["an"].nunique().sort_values()

judet
Alba               11
Arad               11
Arges              11
Bacau              11
Bihor              11
Bistrita-Nasaud    11
Botosani           11
Braila             11
Brasov             11
Buzau              11
Calarasi           11
Caras-Severin      11
Cluj               11
Constanta          11
Covasna            11
Dambovita          11
Dolj               11
Galati             11
Giurgiu            11
Gorj               11
Harghita           11
Hunedoara          11
Ialomita           11
Iasi               11
Ilfov              11
Maramures          11
Mehedinti          11
Mures              11
Neamt              11
Olt                11
Prahova            11
Salaj              11
Satu Mare          11
Sibiu              11
Suceava            11
Teleorman          11
Timis              11
Tulcea             11
Valcea             11
Vaslui             11
Vrancea            11
Name: an, dtype: int64

### 4. Agricultural Employment Change Indicators

In [35]:
df_euro_nuts3 = df_euro_nuts3.sort_values(["judet", "an"]).copy()

df_euro_nuts3["schimbare_pondere_ocupati_aff_eurostat"] = (
    df_euro_nuts3.groupby("judet")["pondere_ocupati_aff_eurostat"].diff()
)

df_euro_nuts3["schimbare_ocupati_aff_eurostat_pct"] = (
    df_euro_nuts3.groupby("judet")["ocupati_aff_eurostat_nr"].pct_change() * 100
)

In [36]:
df_euro_nuts3[[
    "schimbare_pondere_ocupati_aff_eurostat",
    "schimbare_ocupati_aff_eurostat_pct"
]].isna().sum()

nace_r2
schimbare_pondere_ocupati_aff_eurostat    41
schimbare_ocupati_aff_eurostat_pct        41
dtype: int64

In [37]:
df_euro_nuts3[[
    "pondere_ocupati_aff_eurostat",
    "schimbare_pondere_ocupati_aff_eurostat",
    "schimbare_ocupati_aff_eurostat_pct"
]].describe().T

,count,mean,std,min,25%,50%,75%,max
nace_r2,,,,,,,,
pondere_ocupati_aff_eurostat,451.0,28.612497,15.815452,2.730439,14.528335,26.848951,41.541864,66.144914
schimbare_pondere_ocupati_aff_eurostat,410.0,-1.004482,2.510777,-13.004259,-2.255219,-0.899782,0.464342,8.567769
schimbare_ocupati_aff_eurostat_pct,410.0,-2.880482,15.261112,-49.952637,-11.200993,-4.302700,3.668930,81.005222


In [38]:
euro_labour_vars = df_euro_nuts3[
    [
        "judet",
        "an",
        "ocupati_aff_eurostat_mii",
        "ocupati_total_eurostat_mii",
        "ocupati_aff_eurostat_nr",
        "ocupati_total_eurostat_nr",
        "pondere_ocupati_aff_eurostat",
        "schimbare_pondere_ocupati_aff_eurostat",
        "schimbare_ocupati_aff_eurostat_pct"
    ]
].copy()

In [39]:
euro_labour_vars.shape

(451, 9)

In [40]:
euro_labour_vars.duplicated(subset=["judet", "an"]).sum()

np.int64(0)

In [41]:
euro_labour_vars.isna().sum()

nace_r2
judet                                      0
an                                         0
ocupati_aff_eurostat_mii                   0
ocupati_total_eurostat_mii                 0
ocupati_aff_eurostat_nr                    0
ocupati_total_eurostat_nr                  0
pondere_ocupati_aff_eurostat               0
schimbare_pondere_ocupati_aff_eurostat    41
schimbare_ocupati_aff_eurostat_pct        41
dtype: int64

### 5. Export of Processed Eurostat Data

In [42]:
euro_labour_vars = df_euro_nuts3[
    [
        "judet",
        "an",
        "ocupati_aff_eurostat_mii",
        "ocupati_total_eurostat_mii",
        "ocupati_aff_eurostat_nr",
        "ocupati_total_eurostat_nr",
        "pondere_ocupati_aff_eurostat",
        "schimbare_pondere_ocupati_aff_eurostat",
        "schimbare_ocupati_aff_eurostat_pct"
    ]
].copy()

In [43]:
# Save the processed Eurostat labour dataset
output_path = "../../data/processed/eurostat_agricultural_employment.csv"

euro_labour_vars.to_csv(
    output_path,
    index=False
)

print(f"Processed Eurostat data saved to: {output_path}")
print(f"Final dataset shape: {euro_labour_vars.shape}")

Processed Eurostat data saved to: ../../data/processed/eurostat_agricultural_employment.csv
Final dataset shape: (451, 9)
